In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

bronze_table = "ecom_external_catalog.retail_sales_dev.sales_bronze"
silver_table = "ecom_external_catalog.retail_sales_dev.sales_silver"
checkpoint_path = "/Volumes/ecom_external_catalog/retail_sales_dev/pipeline_state/silver"


def merge_silver_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    latest_per_sale = Window.partitionBy("sale_id").orderBy(
        F.col("ingested_at").desc(),
        F.col("source_file").desc(),
    )

    updates = (
        batch_df
        .withColumn("sale_date", F.expr("try_cast(sale_date AS DATE)"))
        .withColumn("quantity", F.expr("try_cast(quantity AS INT)"))
        .withColumn("unit_price", F.expr("try_cast(unit_price AS DECIMAL(12,2))"))
        .filter(F.col("quantity") > 0)
        .withColumn("row_num", F.row_number().over(latest_per_sale))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    target = DeltaTable.forName(batch_df.sparkSession, silver_table)

    (
        target.alias("target")
        .merge(updates.alias("source"), "target.sale_id = source.sale_id")
        .whenMatchedUpdateAll(
            condition="""
                source.ingested_at > target.ingested_at
                OR (
                    source.ingested_at = target.ingested_at
                    AND source.source_file > target.source_file
                )
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


query = (
    spark.readStream.table(bronze_table)
    .writeStream
    .foreachBatch(merge_silver_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

silver = spark.table(silver_table)
print(f"Silver row count: {silver.count()}")
display(silver.orderBy("sale_id"))

In [0]:
silver = spark.table("ecom_external_catalog.retail_sales_dev.sales_silver")

print(f"Silver row count: {silver.count()}")
display(
    silver.where("sale_id IN ('S001', 'S007')")
    .select("sale_id", "quantity", "source_file")
    .orderBy("sale_id")
)